# Классификация текста 
## Ссылки

https://www.youtube.com/watch?v=AwdawjHJstU - Классификация текста | Обработка естественного языка</br>
https://www.youtube.com/watch?v=55Iyei3bkKk&list=PLtPJ9lKvJ4ohZpMV9Ml-DPtMSXPFNl6Sz</br>

https://www.youtube.com/watch?v=-2O0ODmnw1o - NLP cookbook: анализируем тексты на Python с минимальными знаниями о машинном обучении

# 0. Импорты

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import nltk
from nltk.corpus import stopwords
import string
import re
import numpy as np
from tqdm import tqdm

# 1. Загрузка датасета

In [2]:
df = pd.read_json("data/data.txt", lines=True)
df

,text,tags,schema_name,table_name
0,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47923
1,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710
2,"2 2 2 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710
3,"3 3 3 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710
4,"4 4 4 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710
...,...,...,...,...
18419,996 996 982 20231117_110117.jpg 2023-11-17 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18420,997 997 983 20231121_100442.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18421,998 998 984 20231121_095847.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18422,99 99 101 117 4.jpg 2023-11-16 00:00:00+00 13:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876


In [3]:
final_df = df.copy()

In [10]:
final_df['full_table_name'] = final_df['schema_name'] + '.' + final_df['table_name']
final_df['class_name'] =  final_df['tags'].apply(lambda x: '/'.join(x))
final_df['class_id'] = pd.factorize(final_df['class_name'])[0]
final_df

,text,tags,schema_name,table_name,full_table_name,class_name,class_id
0,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47923,_46379._47923,МКА. Геоданные/440 Особо охраняемые природные ...,0
1,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,_46379._58710,МКА. Геоданные/440 Особо охраняемые природные ...,1
2,"2 2 2 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,_46379._58710,МКА. Геоданные/440 Особо охраняемые природные ...,1
3,"3 3 3 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,_46379._58710,МКА. Геоданные/440 Особо охраняемые природные ...,1
4,"4 4 4 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,_46379._58710,МКА. Геоданные/440 Особо охраняемые природные ...,1
...,...,...,...,...,...,...,...
18419,996 996 982 20231117_110117.jpg 2023-11-17 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,_46379._47876,МКА. Геоданные/440 Особо охраняемые природные ...,20
18420,997 997 983 20231121_100442.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,_46379._47876,МКА. Геоданные/440 Особо охраняемые природные ...,20
18421,998 998 984 20231121_095847.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,_46379._47876,МКА. Геоданные/440 Особо охраняемые природные ...,20
18422,99 99 101 117 4.jpg 2023-11-16 00:00:00+00 13:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,_46379._47876,МКА. Геоданные/440 Особо охраняемые природные ...,20
